# Notebook 5: Sequential Testing and Group Sequential Methods

## Overview
This notebook addresses a critical practical problem: **the "peeking problem."** Many companies monitor A/B test results in real-time and stop early when they see promising results. This inflates Type I error unless we use proper sequential testing methods.

### Learning Objectives
- Understand why continuous monitoring inflates Type I error
- Learn how O'Brien-Fleming and Pocock boundaries work
- Implement sequential testing in practice
- Compare fixed-sample vs sequential designs

### The Core Problem
If you run an A/B test designed for 10,000 customers but check the result every 1,000 customers, your true Type I error is NOT 5%—it's much higher. Why? Because each peek is a hypothesis test, and if you do many tests, you'll eventually see p < 0.05 by chance alone.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try plotly
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb05', exist_ok=True)


In [ ]:
# Load cleaned data
data_path = Path("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"Segments: {df['segment'].value_counts()}")

## Concept: The "Peeking Problem"

### The Core Issue
When you **continuously monitor** an A/B test and stop early when you see a favorable result, you're actually conducting multiple hypothesis tests. Each peek is an opportunity to make a Type I error.

### Mathematical Problem
- Single test at fixed sample size: Type I error = α = 0.05
- Two peeks at 50% and 100% of planned sample: Type I error ≈ 0.08 to 0.10 (depending on design)
- **Five peeks**: Type I error can exceed 0.15
- **Twenty peeks**: Type I error can exceed 0.20

This happens because the test statistics across peeks are **correlated but not perfectly correlated**, creating an opportunity to observe p < 0.05 by chance.

### Real-World Example
A company plans a 10,000-customer test with α = 0.05. Instead of waiting, they check:
- After 1,000 customers: p = 0.08 (not significant, but improving)
- After 2,000 customers: p = 0.06 (trending)
- After 3,000 customers: p = 0.04 (stop! declare victory!)

But they never would have seen p < 0.05 if they'd run all 10,000. They lucked out in the multiple comparisons.

### Solution: Sequential Testing
Use **pre-defined boundaries** that adjust the significance level at each peek to maintain overall Type I error control. The key methods are:

1. **O'Brien-Fleming**: Conservative early, liberal late (requires strong evidence early)
2. **Pocock**: Constant boundary (same threshold at each look)
3. **Spending functions**: Flexible allocation of alpha across looks (Lan-DeMets)

In [ ]:
def simulate_fixed_sample_test(null_true=True, n_final=5000, conversion_rate_1=0.05, conversion_rate_2=0.05):
    """
    Simulate a fixed-sample A/B test, computing test statistics at multiple points.
    
    Parameters:
    -----------
    null_true : bool
        Whether the null hypothesis is truly true
    n_final : int
        Final sample size per group
    conversion_rate_1 : float
        True conversion rate for group 1
    conversion_rate_2 : float
        True conversion rate for group 2
    
    Returns:
    --------
    array : p-values at each peek (at 25%, 50%, 75%, 100% of target sample size)
    """
    peeks = np.array([0.25, 0.50, 0.75, 1.0])
    peek_sizes = (peeks * n_final).astype(int)
    
    p_values = []
    
    for peek_n in peek_sizes:
        # Generate data
        if null_true:
            conv1 = np.random.binomial(1, 0.05, peek_n)
            conv2 = np.random.binomial(1, 0.05, peek_n)
        else:
            conv1 = np.random.binomial(1, conversion_rate_1, peek_n)
            conv2 = np.random.binomial(1, conversion_rate_2, peek_n)
        
        # Two-proportion z-test
        p1, p2 = conv1.mean(), conv2.mean()
        se = np.sqrt(p1*(1-p1)/peek_n + p2*(1-p2)/peek_n)
        
        if se == 0:
            z_stat = 0
        else:
            z_stat = (p1 - p2) / se
        
        p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        p_values.append(p_val)
    
    return np.array(p_values)

# Simulate 10,000 null hypothesis tests under continuous monitoring (4 peeks)
print("Simulating 10,000 A/B tests with continuous monitoring...\n")

num_sims = 10000
peeks_matrix = np.zeros((num_sims, 4))

for i in range(num_sims):
    peeks_matrix[i, :] = simulate_fixed_sample_test(null_true=True, n_final=5000)

# Count how many tests would reject at each peek (p < 0.05)
rejections_per_peek = np.sum(peeks_matrix < 0.05, axis=0) / num_sims
ever_rejected = np.sum(np.any(peeks_matrix < 0.05, axis=1)) / num_sims

print("Type I Error by Peek Point:")
print("=" * 50)
for i, pct in enumerate([25, 50, 75, 100]):
    print(f"  After {pct}% of planned sample: {rejections_per_peek[i]:.3f}")

print(f"\nEver rejected (stopped early): {ever_rejected:.3f}")
print(f"\nTarget Type I error: 0.050")
print(f"Actual Type I error (any peek): {ever_rejected:.3f}")
print(f"\n*** Type I error inflation: {ever_rejected / 0.05:.1f}x ***")

In [ ]:
# Plot p-value trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: P-value trajectories
ax = axes[0]
peek_labels = [25, 50, 75, 100]

# Plot first 100 simulations as thin lines
for i in range(100):
    ax.plot(peek_labels, peeks_matrix[i, :], alpha=0.1, color='gray')

# Plot mean trajectory
mean_pvals = peeks_matrix.mean(axis=0)
ax.plot(peek_labels, mean_pvals, color='red', linewidth=2.5, label='Mean p-value')

# Add significance threshold
ax.axhline(0.05, color='black', linestyle='--', linewidth=2, label='α = 0.05')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

ax.set_xlabel('% of Planned Sample Size', fontsize=11)
ax.set_ylabel('p-value', fontsize=11)
ax.set_title('P-value Trajectories Under Continuous Monitoring\n(H₀ True)', fontsize=12, fontweight='bold')
ax.set_ylim(-0.01, 0.15)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Right: Rejection rate by peek
ax = axes[1]
rejection_pcts = rejections_per_peek * 100
bars = ax.bar(peek_labels, rejection_pcts, color=['lightcoral' if x > 5 else 'lightblue' for x in rejection_pcts], 
              edgecolor='black', linewidth=1.5)

ax.axhline(5, color='black', linestyle='--', linewidth=2, label='Target = 5%')
ax.set_xlabel('% of Planned Sample Size', fontsize=11)
ax.set_ylabel('Rejection Rate (%)', fontsize=11)
ax.set_title('Type I Error by Peek Point', fontsize=12, fontweight='bold')
ax.set_ylim(0, 15)
ax.legend(fontsize=10)

# Add value labels
for i, (bar, pct) in enumerate(zip(bars, rejection_pcts)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_peeking_problem.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPeeking problem visualization saved.")

## O'Brien-Fleming Spending Function

### The Idea
The O'Brien-Fleming method uses an **alpha spending function** that is:
- **Conservative early** (high threshold, hard to declare victory)
- **Liberal late** (low threshold, easier to declare victory)

This makes sense: early data has more uncertainty, so we require stronger evidence. Later data has accumulated and is more certain, so we can relax the threshold.

### The Formula
For K looks, the cumulative alpha spent at look k is:

**α_spend(t) = 2 × (1 - Φ(z_α/2 / √t))**

Where t = k/K (fraction of total looks completed).

### Example
For 4 looks with α = 0.05:
- Look 1 (25%): Boundary z = 2.413 (p ≈ 0.0158) — strong evidence needed
- Look 2 (50%): Boundary z = 2.050 (p ≈ 0.0404) — still stringent
- Look 3 (75%): Boundary z = 1.846 (p ≈ 0.0648) — relaxing
- Look 4 (100%): Boundary z = 1.960 (p ≈ 0.0500) — standard 0.05

This maintains overall Type I error at 5% while allowing early stopping.

In [ ]:
def obrien_fleming_boundary(k, K, alpha=0.05):
    """
    Calculate O'Brien-Fleming boundary for look k out of K.
    
    Parameters:
    -----------
    k : int
        Current look (1, 2, ..., K)
    K : int
        Total number of looks
    alpha : float
        Overall Type I error rate
    
    Returns:
    --------
    float : Critical z-value for this look
    """
    t = k / K  # Information time
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_critical = z_alpha / np.sqrt(t)
    return z_critical

def pocock_boundary(k, K, alpha=0.05):
    """
    Calculate Pocock boundary (constant threshold adjusted for repeated testing).
    
    For Pocock, all looks have the same p-value threshold, but it's more stringent than 0.05.
    """
    # Pocock uses a fixed p-value threshold across all looks
    # For K looks, the boundary is approximately: p_threshold = alpha / (0.5 + 0.25*ln(K))
    # Here we use a standard approximation
    
    if K == 1:
        return stats.norm.ppf(1 - alpha / 2)
    elif K == 2:
        p_threshold = alpha / 2.045
    elif K == 3:
        p_threshold = alpha / 2.289
    elif K == 4:
        p_threshold = alpha / 2.447
    elif K == 5:
        p_threshold = alpha / 2.562
    else:
        # General approximation
        p_threshold = alpha / (0.5 + 0.25 * np.log(K))
    
    z_critical = stats.norm.ppf(1 - p_threshold / 2)
    return z_critical

# Calculate boundaries for 4 looks
K = 4
looks = np.arange(1, K + 1)

print("O'Brien-Fleming Boundaries (K=4 looks, α=0.05):")
print("=" * 60)
print(f"{'Look':<6} {'Information %':<15} {'Z Critical':<12} {'p-value':<12}")
print("-" * 60)

of_boundaries = []
pocock_boundaries = []

for k in looks:
    z_of = obrien_fleming_boundary(k, K, alpha=0.05)
    p_of = 2 * (1 - stats.norm.cdf(z_of))
    z_poc = pocock_boundary(k, K, alpha=0.05)
    p_poc = 2 * (1 - stats.norm.cdf(z_poc))
    
    of_boundaries.append(z_of)
    pocock_boundaries.append(z_poc)
    
    info_pct = (k / K) * 100
    print(f"{k:<6} {info_pct:<15.0f} {z_of:<12.3f} {p_of:<12.4f}")

print(f"
Pocock Boundaries (constant, K=4 looks, α=0.05):")
print("=" * 60)
print(f"{'Look':<6} {'Information %':<15} {'Z Critical':<12} {'p-value':<12}")
print("-" * 60)

for k, z_poc in zip(looks, pocock_boundaries):
    p_poc = 2 * (1 - stats.norm.cdf(z_poc))
    info_pct = (k / K) * 100
    print(f"{k:<6} {info_pct:<15.0f} {z_poc:<12.3f} {p_poc:<12.4f}")

In [ ]:
# Plot O'Brien-Fleming vs Pocock vs Fixed Sample
fig, ax = plt.subplots(figsize=(11, 6))

info_pct = np.array([0.25, 0.5, 0.75, 1.0]) * 100

# Plot boundaries
ax.plot(info_pct, of_boundaries, marker='o', markersize=8, linewidth=2.5, 
        label='O\'Brien-Fleming', color='steelblue')
ax.plot(info_pct, pocock_boundaries, marker='s', markersize=8, linewidth=2.5, 
        label='Pocock', color='darkorange')

# Fixed sample threshold (horizontal line)
fixed_threshold = stats.norm.ppf(1 - 0.05 / 2)
ax.axhline(fixed_threshold, color='red', linestyle='--', linewidth=2, 
           label='Fixed Sample (1 look)')

ax.set_xlabel('Information Time (% of Sample)', fontsize=12)
ax.set_ylabel('Critical Z-Value', fontsize=12)
ax.set_title('Sequential Testing Boundaries: O\'Brien-Fleming vs Pocock', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(1.5, 2.5)

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_sequential_boundaries.png', dpi=300, bbox_inches='tight')
plt.show()

print("Boundaries visualization saved.")

## Apply Sequential Testing to Hillstrom Data

We'll simulate as if the Hillstrom data arrived in 5 sequential batches. At each batch, we calculate the z-statistic for the conversion rate difference (Mens Email vs No Email) and compare to the boundaries.

In [ ]:
# Prepare data for sequential analysis
# Split into 5 sequential looks
K_seq = 5
df_sorted = df.sort_values('recency').reset_index(drop=True)
n_total = len(df_sorted)
n_per_look = n_total // K_seq

print(f"Total observations: {n_total}")
print(f"Approximate per look: {n_per_look}\n")

# Track metrics at each look
looks_seq = np.arange(1, K_seq + 1)
z_statistics = []
p_values = []
info_times = []
n_cumulative = []

for k in looks_seq:
    # Get cumulative data up to this look
    n_cum = min(k * n_per_look, n_total)
    df_cum = df_sorted.iloc[:n_cum]
    
    # Filter to Mens Email and No Email
    mens_data = df_cum[df_cum['segment'] == "Mens E-Mail"]
    control_data = df_cum[df_cum['segment'] == "No E-Mail"]
    
    # Calculate conversion rates
    p1 = mens_data['conversion'].mean()
    p2 = control_data['conversion'].mean()
    n1, n2 = len(mens_data), len(control_data)
    
    # Z-test
    p_pooled = (mens_data['conversion'].sum() + control_data['conversion'].sum()) / n_cum
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
    
    if se > 0:
        z_stat = (p1 - p2) / se
    else:
        z_stat = 0
    
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    info_time = k / K_seq
    
    z_statistics.append(z_stat)
    p_values.append(p_val)
    info_times.append(info_time)
    n_cumulative.append(n_cum)
    
    print(f"Look {k}: N={n_cum}, Men's Conv Rate={p1:.4f}, Control Conv Rate={p2:.4f}")
    print(f"        Z-statistic={z_stat:.3f}, p-value={p_val:.4f}\n")

# Calculate boundaries for K_seq looks
of_boundaries_seq = [obrien_fleming_boundary(k, K_seq) for k in looks_seq]
pocock_boundaries_seq = [pocock_boundary(k, K_seq) for k in looks_seq]

In [ ]:
# Visualize sequential testing applied to Hillstrom data
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Top plot: Z-statistics vs boundaries
ax = axes[0]

ax.plot(looks_seq, z_statistics, marker='o', markersize=10, linewidth=2.5, 
        label='Observed Z-statistic', color='black', zorder=5)

ax.plot(looks_seq, of_boundaries_seq, marker='o', markersize=8, linewidth=2, 
        label='O\'Brien-Fleming Boundary', color='steelblue', linestyle='--')
ax.plot(looks_seq, pocock_boundaries_seq, marker='s', markersize=8, linewidth=2, 
        label='Pocock Boundary', color='darkorange', linestyle='--')

# Fixed sample boundary
fixed_z = stats.norm.ppf(1 - 0.05 / 2)
ax.axhline(fixed_z, color='red', linestyle=':', linewidth=2, label='Fixed Sample Boundary')
ax.axhline(-fixed_z, color='red', linestyle=':', linewidth=2)

ax.axhline(0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

ax.set_xlabel('Look Number', fontsize=11)
ax.set_ylabel('Z-Statistic', fontsize=11)
ax.set_title('Sequential Analysis: Z-statistic vs Boundaries\n(Men\'s Email vs No Email - Conversion)', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3)
ax.set_xticks(looks_seq)

# Bottom plot: Information time and decisions
ax = axes[1]

colors = []
decisions = []

for i, (z, of_bound) in enumerate(zip(z_statistics, of_boundaries_seq)):
    if abs(z) > of_bound:
        colors.append('red')
        decisions.append('STOP - Reject H₀')
    else:
        colors.append('lightblue')
        decisions.append('Continue')

bars = ax.bar(looks_seq, np.abs(z_statistics), color=colors, edgecolor='black', linewidth=1.5)
ax.plot(looks_seq, of_boundaries_seq, marker='o', markersize=8, linewidth=2, 
        color='darkred', linestyle='--', label='O\'Brien-Fleming Boundary')

ax.set_xlabel('Look Number', fontsize=11)
ax.set_ylabel('|Z-Statistic|', fontsize=11)
ax.set_title('Decision Rule: Stop if |Z| Exceeds Boundary', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='y')
ax.set_xticks(looks_seq)

# Add decision labels
for i, (bar, decision) in enumerate(zip(bars, decisions)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
            decision, ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_sequential_hillstrom_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Sequential analysis visualization saved.")

In [ ]:
# Create summary comparison table
summary_data = []

for i, k in enumerate(looks_seq):
    summary_data.append({
        'Look': k,
        'Cumulative N': n_cumulative[i],
        'Z-Statistic': f"{z_statistics[i]:.3f}",
        'p-value': f"{p_values[i]:.4f}",
        'OF Boundary': f"{of_boundaries_seq[i]:.3f}",
        'OF Decision': 'STOP' if abs(z_statistics[i]) > of_boundaries_seq[i] else 'Continue',
        'Pocock Decision': 'STOP' if abs(z_statistics[i]) > pocock_boundaries_seq[i] else 'Continue'
    })

summary_table = pd.DataFrame(summary_data)

print("\n=== SEQUENTIAL TESTING SUMMARY ===\n")
print(summary_table.to_string(index=False))

# Save summary
summary_table.to_csv('../data/outputs/nb05/nb05_sequential_results.csv', index=False)

print("\n\nInterpretation:")
print("-" * 70)
print("OF = O'Brien-Fleming (conservative early, liberal late)")
print("Pocock = Constant boundary (same p-value at all looks)")
print("\nWith O'Brien-Fleming:")
print("  - Early looks require strong evidence (higher Z threshold)")
print("  - Later looks have lower threshold, allowing earlier stopping")
print("  - Overall Type I error still controlled at 5%")

## Key Takeaways on Sequential Testing

### When to Use Sequential Testing
1. **Online experiments**: Results stream in gradually, desire to stop early if results are clear
2. **High cost of uncertainty**: When running the test is expensive (customer acquisition, computing)
3. **Ethical studies**: Medical trials where stopping early saves patients from ineffective treatments
4. **Business deadlines**: Need results faster than fixed-sample design allows

### When Fixed-Sample is Better
1. **Small data volume**: Few observations, limited opportunity to look
2. **Off-line analysis**: Data arrives in batch, can wait for pre-planned analysis
3. **Regulatory requirements**: Some contexts require pre-registered, fixed-sample designs

### Best Practices
1. **Pre-register the design**: Decide number of looks and boundaries BEFORE collecting data
2. **Use O'Brien-Fleming**: More powerful than Pocock for fixed sample size
3. **Monitor power**: Ensure you still have ~80% power to detect meaningful effect
4. **Consider adaptive designs**: Lan-DeMets spending functions allow flexible look times
5. **Be transparent**: Report that sequential testing was used and what boundaries were applied

### Common Mistakes
1. **Ignoring correlation**: Believing each peek is independent (it's not)
2. **Changing the boundary**: "Just this once" checking at unplanned times
3. **Multiple testing without adjustment**: Testing multiple metrics without correction
4. **Not adjusting for early stopping**: Reporting CI/estimates as if fixed sample